# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型 |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** 全体の図・部品の役割・管理画面・Android アプリ・用語集 |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |
| [17-cloudflare](17-cloudflare.ipynb) | Cloudflare に載せるときの計画(書くだけ・当てていない) |
| [18-room-shape-research](18-room-shape-research.ipynb) | 写真から部屋の形を作る(調査の報告) |
| [19-security-fixes-2026-10-06](19-security-fixes-2026-10-06.ipynb) | 2026-10-05 の診断の反映(直したもの・確かめたこと・残り) |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |
| [17-cloudflare](17-cloudflare.ipynb) | Cloudflare に載せるときの計画(書くだけ・当てていない) |
| [18-room-shape-research](18-room-shape-research.ipynb) | 写真から部屋の形を作る(調査の報告) |

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()


# 19. 診断(2026-10-05)の反映(2026-10-06)

出典: 利用者から受け取った診断の報告(`security-review-2026-10-05.md`。リポジトリには入れていない)。**値(氏名・IP・メール)は書かない。**
確かめた場所は検証機(192.168.3.107。機密なしのコピー)と LDPlayer。本番には触れていない(当てるのは利用者)。

## 1. 直したもの

| 番号 | 何を | どう直したか | 確かめ方 |
|---|---|---|---|
| **W-54**(低) | 取り消した仮アカウントでも見取り図が取れた | `api/floor-image.php` が錠を見る前に `km_map_guest_verify()` で表から確かめ直す | 検証機で、錠を掛けて仮アカウントのセッションを作り、取り消す前 200 → **取り消した後 403**(同じセッション)。設定ファイルは元に戻した(md5 一致) |
| **A-35**(低) | 停止されたアカウントで、端末の地図の氏名を落とさなかった | `logto_me.php` の 401・403 を「サーバーが断った」(`authorizationDenied`)として通信の不調と分ける。断られたら地図の氏名・スタッフ限定の地点を落とし、「オフラインで入る」の記録も消して使わない。オフラインで開いている間の確かめ直しでも閉じる | JVM の試験(`ReviewFixes1005Test`)。**実機は手順 B・C(利用者のサインインが要る)** |
| 情報: OPcache | 地図の錠・パスワード・配信の停止が最大 2 秒効かない | 設定ファイルを書いた直後に `opcache_invalidate`(`lib/opcache.php`) | check.php |
| 情報: 版番号 | 配布物の版番号(versionCode)が手入力だけ | APK の中(AndroidManifest.xml のバイナリ XML)から読み、入力が空ならその値・違えば断る(`lib/apk-version.php`。zip 拡張が無いので自前で読む。全体は読まない) | 作った APK 4 本で正しく読めた(約 1ms・0.7MB)。check.php で作り物の APK(deflate・格納・壊れたもの) |
| 依存のイメージ | 古い digest のまま | nginx・Mailpit・MariaDB・Postgres・phpMyAdmin・php:8.4-apache・composer:2・node:24 を今の digest に上げた(**Logto は据え置き**) | 検証機で取り込み・作り直し・全サービス healthy・公開ページ・地図・管理画面・アプリの地図が 200 |
| okio | APK に 3.2.0(CVE-2023-3635) | 制約で 3.9.1 に揃えた | `gradlew dependencies` で 3.2.0 → 3.9.1、両フレーバーの試験、LDPlayer で起動と Logto のサインイン画面まで |
| check.php | 1 つの例外で以降が走らない | 組ごとに例外を受け止め、その組を失敗として数えて次へ | わざと壊した写しで、残りの組が走り exit 1 になることを確認 |
| push 前の検査 | メールの検出が 6 ドメインだけ・`-ScanAll` が Windows 以外で止まる | どのドメインでも拾う(見本・自分のサービスの宛先は除く。知らせるだけ)・`Get-Item -Force` | 8 つの例で、拾う・除くが期待どおり |

## 2. 確かめた結果(未検証だったもの)

| 項目 | 結果 |
|---|---|
| Logto(1.43.0)・Soketi のイメージの脆弱性(手順 D) | Logto: HIGH・CRITICAL 合わせて 83 件(CRITICAL 2)。Soketi(作り直した後): 71 件(CRITICAL 0) |
| Logto の JWKS の `alg` | 付いている(ES384)。php-jwt は鍵を使える |
| ID トークンの有効期間 | ネイティブ・サイト用とも設定は空(Logto の既定で動いている) |
| `X-Real-IP` の上書き | 偽の `X-Real-IP`・`X-Forwarded-For` を付けても、記録は本当の送り元。nginx が `$remote_addr` で上書きしている |
| `pipefail` | ホストの sh(dash 0.5.12)には**無い**(実測)。全体で入れると `… | head` が落ちるので入れない。バックアップのダンプはファイルへ直に書き、終端(Dump completed)を確かめる作りで、パイプで失敗を見逃す形ではない |
| 読む: かんばん・進捗表・予定表 | 権限(書き込みは guard がまとめて見る)・CSRF・エスケープ・SQL とも問題なし。予定表の色は許可した値だけ |
| 読む: `app-map-convert.php`・`app-map-sync.php` | SQL は固定の文字列か、列名を決まった一覧との共通部分に絞ったものだけ。取り込みは 20MB まで |
| CVE の中身(CRITICAL のもの) | **gosu の CVE-2025-68121**(CVSS 10)は Go の TLS の再開の不具合。gosu は TLS を使わないので当たらない。**twig**(phpMyAdmin)はテンプレートを攻撃者が書けることが前提で、phpMyAdmin のテンプレートは固定・IP の許可と Logto の門の内側。**perl の CVE-2026-13221** は 65535 を超える分岐の正規表現でだけ起きる |

### イメージを上げる前と後(HIGH + CRITICAL の件数。Trivy)

| イメージ | 診断のとき | 上げた後 |
|---|---|---|
| phpMyAdmin | 287(CRITICAL 20) | 213(CRITICAL 8。うち 6 は同梱の twig。上流待ち) |
| web(php:8.4-apache) | 229(CRITICAL 14) | 194(CRITICAL 2) |
| nginx | 18 | 2 |
| Mailpit | 2 | 0 |
| MariaDB | 24(CRITICAL 1) | 22(CRITICAL 1。gosu) |
| Postgres | 31(CRITICAL 1) | 22(CRITICAL 1。gosu) |
| Soketi(node) | 69(CRITICAL 3) | 71(CRITICAL 0) |

## 3. 残り(利用者)

| 項目 | すること |
|---|---|
| **W-55**(中) | GitHub Support に PR の参照(#1〜#6)の削除を頼む。下書きは `Website/github-support-W-55.md`(GitHub の文書を読み直して書いた。**教職員の氏名を主な理由にする** —— 資格情報を替えれば済むものには Support は手を貸さない) |
| 手順 A(A-29)・B(A-32)・C(A-35) | LDPlayer のアプリでサインインが要る。**検証機には教職員の組織が無い**(検証機向けの APK は `staffOrgId=none`)ので、B・C は「管理者の権限を持つ試験用の利用者」で代わりに確かめる |
| Logto 1.43.0 → 1.44.0 | 起動時に DB を移行するので、上げるかを決める(上げるなら先に検証機で) |
| 署名の鍵の入れ替え(`AppUpdater`) | 今は「署名が同じ」ことを求めるので、鍵を替えると正当な更新も拒む。**鍵を替える予定が無ければ今のまま**でよい(替えるときは、前の鍵の系譜を確かめる形に直す) |
| W-44c・W-45・W-52c | 設計どおり(年 20,000 行で止まる・公開一覧の閾値・古いアプリ)として今のままにするかを決める |
| Pixel 7a での実機の残り | 飛ばした(利用者が Pixel 7a を使うため) |

## 4. 本番に当てるには

第 1 部と同じ手順の中に入る(`deploy-to-host.ps1 -Action up -Yes` で web・soketi の作り直しとイメージの取り込みも行われる)。
**先に `free -h` で空きを見る**(イメージの取り込みの間はディスクとメモリを使う。検証機は 2GB で、取り込みを 1 つずつにした)。